# Phase 7a: Classification-prefix embeddings on Colab

Thin wrapper: clones the repo, installs deps, builds the corpus, runs
`trustlayer.classify.vectors` on CUDA, saves each chunk file to Drive as
it lands. No logic lives here — the module owns everything. Interrupt
and re-run any time: completed chunks restore from Drive and are skipped
(resume).

Use a GPU runtime (T4): Runtime → Change runtime type → T4 GPU.

Expected: ~10–15 min for 23,267 chunks on T4. Output: 24 chunk files
(~70 MB total) in Drive at `trustlayer/embeddings-cls/` + a stats JSON.
Download them back to `data/processed/embeddings-cls/` afterwards — the
guide in `docs/colab-phase7.md` has the exact paths.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ctru0009/trustlayer.git /content/trustlayer
%cd /content/trustlayer/python
!pip install -q uv
!uv sync --extra embed --extra spark

In [ ]:
# Spark 4.x needs Java 17+; Colab does not reliably ship one.
!apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
!java -version

In [ ]:
# Colab starts with no data: download AESLC + build the corpus (~2 min).
!uv run --extra spark python ../data/scripts/download_aeslc.py
!uv run --extra spark python -m trustlayer.prep.run

In [ ]:
import shutil
import subprocess
from pathlib import Path

OUT = Path('/content/trustlayer/data/processed/embeddings-cls')
BACKUP = Path('/content/drive/MyDrive/trustlayer/embeddings-cls')
OUT.mkdir(parents=True, exist_ok=True)
BACKUP.mkdir(parents=True, exist_ok=True)

# Restore previously saved chunks so the run resumes where it stopped.
for saved in BACKUP.glob('chunk-*.jsonl'):
    if not (OUT / saved.name).exists():
        shutil.copy(saved, OUT / saved.name)
        print(f'[drive] restored {saved.name}')

# NOTE: the cls-vs-Document sanity check needs the Phase 4 Document
# vectors, which are NOT rebuilt here (another ~12 min). The prompt line
# printed at startup ('Classification' -> 'task: classification | ...')
# plus the startup ValueError if the prompt were missing are the Colab-side
# proof the right prompt applied; the cosine check runs locally after you
# download the chunks (guide step 4).
proc = subprocess.Popen(
    ['uv', 'run', '--extra', 'embed', '--extra', 'spark',
     'python', '-m', 'trustlayer.classify.vectors', '--skip-sanity',
     '--device', 'cuda', '--batch-size', '64'],
    cwd='/content/trustlayer/python', text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
seen = {p.name for p in BACKUP.glob('chunk-*.jsonl')}
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')
    for fresh in OUT.glob('chunk-*.jsonl'):
        if fresh.name not in seen:
            shutil.copy(fresh, BACKUP / fresh.name)
            seen.add(fresh.name)
            print(f'[drive] saved {fresh.name}')
# The module writes embeddings-cls-stats.json next to OUT; back it up too.
stats = OUT.parent / 'embeddings-cls-stats.json'
if stats.exists():
    shutil.copy(stats, BACKUP / stats.name)
    print(f'[drive] saved {stats.name}')
print('exit:', proc.wait())